## ***Input Embeddings*** 

<p align="center">
  <img src="./images/Input-Emdeddings.png" alt="Input Embeddings">
</p>

### What is an Input Embedding?

Input embedding converts each input token into a vector of a fixed dimension.

For example, in the original Transformer, each token is represented as a **512-dimensional vector**.
Example :

***Example :***

```text
"I love AI"
	  ↓
Tokenization
	  ↓
[I, love, AI]
	  ↓
Input Embedding
	  ↓
[512-d vector, 512-d vector, 512-d vector]
```

Each token gets its own 512-dimensional vector.

In [1]:
import torch 
import torch.nn as nn 
import math

In [2]:
class InputEmbeddings(nn.Module):

	def __init__(self, d_model: int, vocab_size: int) -> None:
		super().__init__()
		self.d_model = d_model
		self.vocab_size = vocab_size
		self.embedding = nn.Embedding(vocab_size, d_model)

	def forward(self, x):
		# (batch, seq_len) --> (batch, seq_len, d_model)
		# Multiply by sqrt(d_model) to scale the embeddings according to the paper
		return self.embedding(x) * math.sqrt(self.d_model)


## **Positional Encoding**

<p align="center">
  <img src="./images/Positional-Encoding.png.webp" alt="Positional Encoding">
</p>

<div align="center">

We saw before that the **Embedding layer** converts each token into a vector.

For example, with a `512`-dimensional embedding:

```text
"I love AI"

     ↓

Embedding

     ↓

[
  I     → vector of size 512
  love  → vector of size 512
  AI    → vector of size 512
]
```

### **The Problem**

The Transformer does not know the **position** of each token in the sentence.

```text
I     → position 0
love  → position 1
AI    → position 2
```

### **The Solution**

We add a **Positional Encoding vector** to each token embedding.

The positional vector has the **same size** as the embedding: `512`.

```text
Token Embedding          Positional Encoding

     (512)                     (512)

        ↓                         ↓

        └────────── + ───────────┘

               ↓

             Final vector (512)
```

So:

$$
\text{Input} =
\text{Token Embedding} +
\text{Positional Encoding}
$$

### **Why do we need it?**

It gives the Transformer information about **where each token is located in the sequence**.

**Embedding** → What is the token?

**Positional Encoding** → Where is the token?

</div>


In [3]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # Create a matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # Create a vector of shape (seq_len)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1) || .unsqueeze() so we can have each number on it's own list like [[0],[1],[2],[3],....]
        # Create a vector of shape (d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # (d_model / 2)

        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term) # sin(position * (10000 ** (2i / d_model))
        # Apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term) # cos(position * (10000 ** (2i / d_model))
        # Add a batch dimension to the positional encoding
        pe = pe.unsqueeze(0) # (1, seq_len, d_model)
        # Register the positional encoding as a buffer
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False) # (batch, seq_len, d_model)
        return self.dropout(x)

### 🔑 Key Notes

The original Transformer creates Positional Encoding using **sine and cosine functions**.

$$
PE(pos,2i)=
\sin\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

$$
PE(pos,2i+1)=
\cos\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

**Important:**

* `position` → tells us the position of each token.
* `sin` → used for even dimensions: `0, 2, 4, ...`
* `cos` → used for odd dimensions: `1, 3, 5, ...`
* Each position gets a different vector.
* The Positional Encoding has the same size as the embedding.
* Finally, we add it to the token embeddings.

$$
\boxed{
\text{Input}
=
\text{Embedding}
+
\text{Positional Encoding}
}
$$

> 💡 You don't need to memorize the formula yet. The main idea is that **sin/cos are used to create a unique vector for each position**.


## ***Layer Normalization***

<p align="center">
  <img src="./images/Layer-Normalization.png" alt="Positional Encoding">
</p>


*Layer Normalization* (**LayerNorm**) normalizes the activations of each token to make the values more stable during training.

For example:

```text
Before LayerNorm:
[0.2, 8.5, -3.1, 12.7, ...]

        ↓ LayerNorm

After LayerNorm:
[normalized values]
```

###  **Formula**

First, calculate the mean and variance:

$$
\mu = \text{mean}(x)
$$

$$
\sigma^2 = \text{variance}(x)
$$

Then normalize:

$$
\hat{x} =
\frac{x-\mu}
{\sqrt{\sigma^2+\epsilon}}
$$

Finally, apply learnable parameters:

$$
y = \gamma\hat{x}+\beta
$$

### **Key Notes**

* **LayerNorm** normalizes the activations of each token.
* It works on the **last dimension** (`d_model`).
* `γ` and `β` are **learnable parameters**.
* `ε` is a small value used to avoid division by zero.
* It helps make **training more stable**.

In PyTorch:

```python
nn.LayerNorm(d_model)
```

If:

```text
x.shape = (batch, seq_len, d_model)
```

LayerNorm is applied to:

```text
(batch, seq_len, d_model)
                    ↑
              normalize here
```

> 💡 **Main idea:** LayerNorm keeps the activations more stable so the Transformer can train more effectively.


In [4]:
class LayerNormalization(nn.Module):

    def __init__(self, features: int, eps:float=10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
         # Keep the dimension for broadcasting
        mean = x.mean(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # Keep the dimension for broadcasting
        std = x.std(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # eps is to prevent dividing by zero or when std is very small
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

## **Feed Forward**

<p align="center">
  <img src="./images/feed-forward.png" alt="Positional Encoding">
</p>

<div align="center">


</div>

The **Feed Forward Network (FFN)** is a fully connected neural network used in both the **Encoder** and the **Decoder** of the Transformer.

*Its job is to transform the information that comes from the* **Attention layer**.

---

##  **What does it do?**

The Feed Forward Network consists of **two Linear layers** with a `ReLU` activation between them:

```text
Input
  ↓
Linear 1
  ↓
ReLU
  ↓
Linear 2
  ↓
Output
```

The original Transformer paper defines it as:

$$
\boxed{
FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2
}
$$

---


##  **Dimensions**

In the original Transformer:

$$
d_{model}=512
$$

$$
d_{ff}=2048
$$

The first Linear layer expands the dimension:

```text
512 → 2048
```

The second Linear layer brings it back:

```text
2048 → 512
```

So the complete flow is:

```text
512
 ↓
Linear
 ↓
2048
 ↓
ReLU
 ↓
512
```

### **Why 2048?**

The Feed Forward Network temporarily expands the representation to a larger dimension, giving the model more capacity to learn complex transformations.

---

##  **The Two Weight Matrices**

The paper uses two matrices:

$$
W_1 : 512 \rightarrow 2048
$$

$$
W_2 : 2048 \rightarrow 512
$$

And two biases:

$$
b_1,\ b_2
$$

So:

```text
x
 ↓
W₁ + b₁
 ↓
2048 dimensions
 ↓
ReLU
 ↓
W₂ + b₂
 ↓
512 dimensions
```

---

##  **PyTorch**

In PyTorch, we can implement this using two `Linear` layers:

```python
self.linear_1 = nn.Linear(d_model, d_ff)
self.linear_2 = nn.Linear(d_ff, d_model)
```

With the original Transformer dimensions:

```python
d_model = 512
d_ff = 2048
```

The forward pass is:

```python
x = self.linear_1(x)
x = torch.relu(x)
x = self.linear_2(x)
```

Or simply:

```python
return self.linear_2(torch.relu(self.linear_1(x)))
```

---

##  **Key Notes**

* **FFN = two Linear layers + ReLU.**
* It is used in both the **Encoder and Decoder**.
* The first layer expands:

$$
512 \rightarrow 2048
$$

* The second layer projects back:

$$
2048 \rightarrow 512
$$

* `W₁` and `b₁` belong to the first Linear layer.
* `W₂` and `b₂` belong to the second Linear layer.
* `ReLU` is applied between the two Linear layers.

###  **Main Idea**

> **Attention mixes information between tokens.**

> **Feed Forward transforms the information of each token independently.**

```text
Attention
   ↓
"Which tokens should I pay attention to?"
   ↓
Feed Forward
   ↓
"How should I transform this information?"
```


In [5]:
class FeedForwardBlock(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) # w1 and b1
        self.dropout = nn.Dropout(dropout)       # Dropout turns off some neurons during training to reduce overfitting. 
        self.linear_2 = nn.Linear(d_ff, d_model) # w2 and b2

    def forward(self, x):
        # (batch, seq_len, d_model) --> (batch, seq_len, d_ff) --> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

## **Multi-Head Attention**

You remember that in the `encoder`, we have Multi-Head Attention, which takes the input of the encoder and uses it three times.

One time it is called the *`Query`*, one time it is called the *`Key`*, and one time it is called the *`Value`*.

You can also think of it as duplicating the input three times, or you can simply say that the same input is applied three times. Multi-Head Attention basically works like this: we have our input sequence, which is a matrix of sequence length by d_model. We transform it into three matrices, Q, K, and V, which initially come from the same input in this case because we are talking about the encoder. We will see that in the decoder, it is slightly different.

Then, we multiply these matrices by three different matrices called WQ, WK, and WV. This results in three new matrices, each with dimensions of sequence length by d_model.

We then split these matrices into H smaller matrices, where H is the number of heads we want for this Multi-Head Attention. We split these matrices along the embedding dimension, not along the sequence dimension. This means that each head has access to the full sentence, but it works with a different part of the embedding of each word.

We apply the Attention mechanism to each of these smaller matrices using the formula, which gives us smaller matrices as the results. Then, we combine them back together. So, as the paper says, we concatenate Head 1 up to Head H.

Finally, we multiply the concatenated matrix by WO to get the Multi-Head Attention output, which again is a matrix that has the same dimensions as the input matrix. As you can see in this slide, the output of the Multi-Head Attention is also sequence length by d_model.

<p align="center">
  <img src="./images/Multi-Head-attention.png" width="1000">
</p>

## **What does Multi-Head Attention do?**

In simple terms:

> **Multi-Head Attention allows the model to look at a sentence from multiple perspectives at the same time.**

Each head performs its own attention using **Q, K, and V**, and then the results from all heads are combined.

### **Simple Example**

Suppose we have the sentence:

```text
"The cat is sleeping"
```

We start with the input embeddings:

```text
Input
  ↓
"The"  "cat"  "is"  "sleeping"
```

In **Self-Attention**, the same input is used to create **Q, K, and V**:

```text
                 Input
              /    |    \
             ↓     ↓     ↓
            Q      K      V
```

Then we apply a different linear transformation to each one:

```text
Q × WQ → Q'
K × WK → K'
V × WV → V'
```

Now we have:

```text
Q', K', V'
```

### **Split into Multiple Heads**

Suppose we have **4 heads**:

```text
Q' ──→ Split ──→ Q1  Q2  Q3  Q4
K' ──→ Split ──→ K1  K2  K3  K4
V' ──→ Split ──→ V1  V2  V3  V4
```

Each head performs its own attention:

```text
Head 1 → Attention(Q1, K1, V1)
Head 2 → Attention(Q2, K2, V2)
Head 3 → Attention(Q3, K3, V3)
Head 4 → Attention(Q4, K4, V4)
```

For example, one head might learn a relationship between:

```text
"cat" ←→ "sleeping"
```

while another head might focus on a different relationship in the sentence.

Finally, we combine all the heads:

```text
Head 1 ─┐
Head 2 ─┤
Head 3 ─┤ → Concatenate → WO → Output
Head 4 ─┘
```

### **The Big Picture**

```text
Input
  ↓
Create Q, K, V
  ↓
Apply WQ, WK, WV
  ↓
Split into Heads
  ↓
Attention(Qi, Ki, Vi) for each Head
  ↓
Concatenate all Heads
  ↓
WO
  ↓
Multi-Head Attention Output
```

### **Key Idea**

Think of it like this:

* **Q (Query)** → What am I looking for?
* **K (Key)** → What information do I have?
* **V (Value)** → What information should I take?

And **Multi-Head Attention** does this from **multiple perspectives at the same time**, allowing the model to learn different relationships between words.


In [6]:
class MultiHeadAttentionBlock(nn.Module):

	def __init__(self, d_model: int, h: int, dropout: float):
		super().__init__()
		self.d_model = d_model
		self.h = h 
		assert d_model % h == 0, "d_model is not divisible by h"

		self.d_k = d_model // h 
		self.w_q = nn.Linear(d_model, d_model) # Wq
		self.w_k = nn.Linear(d_model, d_model) # Wk
		self.w_v = nn.Linear(d_model, d_model) # Wv

		self.w_o = nn.Linear(d_model, d_model) # Wo 
		self.dropout = nn.Dropout(dropout)

		@staticmethod
		def attention(query, key, value, mask, dropout: nn.Dropout):
			d_k = query.shape[-1]

			# (Batch, h, Seq_Len, d_k) --> (Batch, h, Seq_Len, Seq_Len)
			attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
			if mask is not None:
				attention_scores.masked_fill(mask == 0, -1e9)

			attention_scores = attention_scores.softmax(dim = -1) # (Batch, h, seq_len, seq_len)

			if dropout is not None :
				attention_scores = dropout(attention_scores)

			return (attention_scores @ value).attention_scores
	
		# Forward Method 
		def Forward(self, q, k, v, mask):
			# mask it just hiad the next token so the 
			query = self.w_q(q)  #(Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model)
			key = self.w_k(k)    #(Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model)
			value = self.w_v(v)  #(Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model)

			# (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, h, d_k) --> (Batch, h, seq_Len, d_k)
			query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)
			key = key.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)
			value = value.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)

			x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

			# (Batch, h, Seq_Len, d_k) --> (Batch, Seq_Len, h, d_k) --> (Batch, Seq_Len, d_model)
			x = x.transpose(1, 2).contiquous().view(x.shape[0], -1, self.h * self.d_k)

			# (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, d_model)
			return self.w_o(x)


## **Residual Connection**


### *What is a Residual Connection?*

A **Residual Connection** is a shortcut connection that passes the original input directly to the output of a layer.

Instead of only using:

```text
Layer(x)
```

we use:

```text
x + Layer(x)
```

### *What does it do?*

The original input `x` is added to the output of the layer:

```text
        ┌──────────────────────┐
        │                      ↓
Input ──┼──→ Layer ──→ Output ──→ Add
        │                      ↑
        └──────────────────────┘
```

So the final output is:

```text
Output = x + Layer(x)
```

### *Why is it important?*

Residual Connections help the model **preserve the original information** while learning new transformations.

They also make it easier for **gradients to flow through deep networks**, which helps the model train more effectively.

### *In Transformers*

Residual Connections are used around important components such as:

```text
Input
  ↓
Multi-Head Attention
  ↓
Add & Norm
  ↓
Feed-Forward Network
  ↓
Add & Norm
  ↓
Output
```

For example:

```text
x → Multi-Head Attention → Attention Output
│                              │
└──────────────────→ Add ←─────┘
                       ↓
                  Layer Norm
```

### *Key Idea*

> **Residual Connection = keep the original input and add it to the layer's output.**

```text
x + Layer(x)
```

This allows the network to **learn new information without losing the original information**.


<p align="center">
  <img src="./images/Residual-Connections.png" width="400">
</p>

In [7]:
class ResidualConnection(nn.Module):

	def __init__(self, dropout : float) -> None:
		super().__init__()
		self.dropout = nn.Dropout(dropout)
		self.norm = LayerNormalization()

	def forward(self, x, sublayer):
		return x * self.dropout(sublayer(self.norm(x)))
	

## **Encoder**

<p align="center">
  <img src="./images/Encoder.png" width="400">
</p>

In [ ]:
class EnoderBlock(nn.Module):

	def __init__(self, self_attenction_block : MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
		super().__init__()
		self.self_attenction_block = self_attenction_block
		self.feed_forward_block = feed_forward_block
		self.residual_connections = nn.ModuleList((ResidualConnection(dropout) for _ in range(2)))

	def forward(self, x, src_mask):
		x = self.residual_connections[0](x, lambda x: self.self_attenction_block(x, x, x, src_mask))
		x = self.residual_connections[1](x, self.feed_forward_block)
		return x 

class Encoder(nn.Module):

	def __init__(self, layers: nn.ModuleList) -> None:
		super().__init__()
		self.layers = layers
		self.norm = LayerNormalization()

	def forward(self, x, mask):
		for layer in self.layers:
			x = layer(x, mask)

		return self.norm(x)

## **Decoder**

<p align="center">
  <img src="./images/Decoder.png" width="400">
</p>

In [10]:
class DecoderBlock(nn.Module):

	def __init__(self, self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float):
		super().__init__()
		self.self_attention_block = self_attention_block
		self.across_attention_block = cross_attention_block
		self.feed_forward_block = feed_forward_block
		self.residual_connections = nn.Module([ResidualConnection(dropout) for _ in range(3)])

	def forward(self, x, encoder_output, src_mask, tgt_mask): # src_mask : the one coming form the encoder , tgt_mask : the one coming form the decoder 
		x = self.residual_connections[0](x, lambda x: self.self_attention_block(x,x,x,tgt_mask))
		x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
		return x 

class Decoder(nn.Module):

	def __init__(self, layers: nn.ModuleList) -> None:
		super().__init__()
		self.layers = layers
		self.norm = LayerNormalization()

	def forward(self, x, encoder_output, srs_mask, tgt_mask):
		for layer in self.layers:
			x = layer(x, encoder_output, srs_mask, tgt_mask)
		return self.norm(x)



## **Linear Layer**

<p align="center">
  <img src="./images/Linear-layer.png" width="400">
</p>

### **What is a Linear Layer?**

A **Linear Layer** is a layer that takes an input and transforms it into a new representation using **weights and a bias**.

The basic formula is:

```text
y = xW + b
```

Where:

* `x` → input
* `W` → weights
* `b` → bias
* `y` → output

### **Simple Example**

Suppose our input has **3 values**:

```text
x = [2, 4, 6]
```

The Linear Layer uses its learned **weights and bias** to transform these values into a new representation.

For example:

```text
Input
  ↓
Linear Layer
  ↓
Output
```

The important thing is that the Linear Layer **changes the representation of the input**.

### **In PyTorch**

A Linear Layer can be created with:

```python
nn.Linear(in_features, out_features)
```

For example:

```python
nn.Linear(3, 2)
```

This means:

```text
Input  →  3 features
Output →  2 features
```

So:

```text
[2, 4, 6]
     ↓
Linear Layer
     ↓
[output1, output2]
```

### **Why is it important?**

A Linear Layer allows the model to **learn how to transform information**.

During training, the model learns the weights and bias so that the output becomes useful for the task.

### **Linear Layers in Transformers**

Linear Layers are used everywhere in Transformers.

For example, in **Multi-Head Attention**, we use Linear Layers to create:

```text
Q = xWQ
K = xWK
V = xWV
```

So the input `x` is transformed into:

```text
Input
  ↓
Linear Layer → Query (Q)

Input
  ↓
Linear Layer → Key (K)

Input
  ↓
Linear Layer → Value (V)
```

### **Key Idea**

> **A Linear Layer learns weights and bias to transform an input representation into a new representation.**

```text
Input → Linear Layer → Output
```

```text
y = xW + b
```


In [11]:
class ProjectionLayer(nn.Module):

	def __init__(self, d_model: int, vocab_size: int) -> None:
		super().__init__()
		self.proj = nn.Linear(d_model, vocab_size)

	def forward(self, x):
		# (Batch, Seq_Len, d_model) --> (Batch, Seq_Len, Vocab_Size)
		return torch.log_softmax(self.proj(x), dim= -1)